# 진짜 미노출 valid로 α 재검증 (`paper/GAP_ANALYSIS.md` §1.3)

지금까지 모든 α 선택(§9의 BEST_ALPHA_V4부터 §14의 "valid 기반" 선택까지)은 **2025년 전체로 학습된
풀 에이전트**를 대상으로 했다 — 2025-10~12(valid로 썼던 구간)도 사실 학습 때 이미 노출된 데이터였다
(§14 한계로 반복 명시됨). walk-forward(다년 검증)는 유니버스 나이 문제로 불가능하다고 판명됐지만,
**같은 2025년 안에서 한 번 엄격하게 train/valid를 나누는 것**(v3, `crypto_hrl_earnhft_colab.ipynb`
§12가 이미 쓴 방법)은 여전히 가능하고 훨씬 저비용이다.

이 노트북은:
1. β=-30을 **2025-01~09만**으로 재학습(seed=42, 배포 모델과 동일 시드로 비교 가능하게)
2. **2025-10~12(이번엔 진짜 미노출)**에서 α를 스윕해 최적값 선택
3. **2026 test는 그 α로 1회만** 실행
4. 기존 배포 모델(2025 전체 학습, 노출된 valid에서 α=0.01 선택, test +5.66%)과 나란히 비교

판정: 이 "진짜 클린" 모델의 test 성적이 기존 배포 모델과 비슷하면 → 지금까지의 결과가 valid 노출
때문에 부풀려진 게 아니라는 강한 근거. 크게 떨어지면 → 지금까지 보고한 성능 상당 부분이 valid
노출에서 온 과최적화라는 뜻 — 정직하게 그렇게 보고해야 함.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 11.4 MB/s eta 0:00:00


## 1. 설정

In [2]:
import warnings
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor

warnings.filterwarnings('ignore')
plt.rcParams['axes.unicode_minus'] = False

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
CRYPTO_DIR = DATA_DIR / 'crypto'
FEAT_DIR   = CRYPTO_DIR / 'features_5m'
MODEL_DIR  = DRIVE_ROOT / 'models'

RL_EMB_H5 = CRYPTO_DIR / 'crypto_rl_embeddings.h5'

EMB_DIM    = 64
TC         = 0.001
BUCKET_SEC = 1800

# --- 기존(노출된) 분할 — 비교 참조용 ---
RL_TRAIN_START = '2025-01-01'
RL_TRAIN_END   = '2025-12-31'
RL_TEST_START  = '2026-01-01'
RL_TEST_END    = '2099-12-31'

# --- 이 노트북의 핵심: 엄격 분할 (v3와 동일 경계) ---
TRUE_TRAIN_END   = '2025-09-30'   # 이 노트북에서 실제로 학습에 쓰는 구간의 끝
TRUE_VALID_START = '2025-10-01'   # 여기부터 12월까지 — 학습에 전혀 안 씀, 진짜 미노출
TRUE_VALID_END   = '2025-12-31'

N_LABELS       = 5
RISK_THRESHOLD = 0.2
LOWPASS_WINDOW = 48

TOP_K = 10
BETA  = -30
SEED  = 42                        # 배포 모델과 동일 시드로 비교 가능하게
POOL_TIMESTEPS_EACH = 100_000

SMOOTH_ALPHAS = [1.0, 0.3, 0.15, 0.05, 0.03, 0.02, 0.01]

DEPLOYED_AGENT_PATH = MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{BETA}.zip'
CLEAN_MODEL_DIR = MODEL_DIR / 'crypto_clean_valid'

assert RL_EMB_H5.exists(), 'crypto_rl_embeddings.h5 없음'
assert DEPLOYED_AGENT_PATH.exists(), f'배포 중인 에이전트 없음: {DEPLOYED_AGENT_PATH}'
print('경로 확인 완료')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


경로 확인 완료


## 2. 데이터 로드

In [3]:
print('crypto_rl_embeddings.h5 로드 중...')
with h5py.File(RL_EMB_H5, 'r') as f:
    symbols_raw = f['symbols'][:]
    ts_raw      = f['timestamps'][:]
    embs_all    = f['embeddings'][:]

symbols_list = [s.decode() for s in symbols_raw]
ts_list      = ts_raw.astype(np.int64).tolist()

emb_index  = {}
sym_ts_set = {}
for i, (s, t) in enumerate(zip(symbols_list, ts_list)):
    emb_index[(s, t // BUCKET_SEC)] = i
    sym_ts_set.setdefault(s, set()).add(t)

symbols  = sorted(set(symbols_list))
N_ASSETS = len(symbols)
print(f'임베딩: {embs_all.shape} | 심볼: {N_ASSETS}개 | 인덱스: {len(emb_index):,}개')


crypto_rl_embeddings.h5 로드 중...
임베딩: (1079473, 64) | 심볼: 44개 | 인덱스: 1,079,473개


In [4]:
print('features_5m parquet에서 RetTarget_6b 로드 중...')
files = sorted(glob.glob(str(FEAT_DIR / '*.parquet')))
assert files, f'피처 parquet 없음: {FEAT_DIR}'

rl_start_ts = pd.Timestamp(RL_TRAIN_START, tz='UTC')
ret_index   = {}

for fp in files:
    sym = Path(fp).stem
    valid_ts = sym_ts_set.get(sym)
    if not valid_ts:
        continue
    df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])
    df = df[df['datetime'] >= rl_start_ts]
    epoch = df['datetime'].values.astype('int64') // 10**9
    y     = df['RetTarget_6b'].values
    for t, r in zip(epoch, y):
        t = int(t)
        if t in valid_ts and not np.isnan(r):
            ret_index[(sym, t // BUCKET_SEC)] = float(r)

all_buckets = sorted(set(t // BUCKET_SEC for t in ts_list))
print(f'수익률 인덱스: {len(ret_index):,}개 | 전체 30분 버킷: {len(all_buckets):,}개')


features_5m parquet에서 RetTarget_6b 로드 중...
수익률 인덱스: 1,079,473개 | 전체 30분 버킷: 26,840개


## 3. CryptoPortfolioEnv

In [5]:
class CryptoPortfolioEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, buckets, symbols, emb_index, embs_all, ret_index,
                 emb_dim=64, tc=0.001):
        super().__init__()
        self.buckets    = buckets
        self.symbols    = symbols
        self.n_assets   = len(symbols)
        self.emb_index  = emb_index
        self.embs_all   = embs_all
        self.ret_index  = ret_index
        self.emb_dim    = emb_dim
        self.tc         = tc

        obs_dim = self.n_assets * emb_dim + self.n_assets
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Box(-3.0, 3.0, shape=(self.n_assets,), dtype=np.float32)

    def _asset_emb(self, bucket):
        embs = []
        for sym in self.symbols:
            idx = self.emb_index.get((sym, bucket))
            embs.append(self.embs_all[idx] if idx is not None else np.zeros(self.emb_dim, dtype=np.float32))
        return np.concatenate(embs)

    def _asset_ret(self, bucket):
        return np.array([self.ret_index.get((sym, bucket), 0.0) for sym in self.symbols], dtype=np.float32)

    def _get_obs(self):
        b = self.buckets[self.t]
        return np.concatenate([self._asset_emb(b), self.weights]).astype(np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t               = 0
        self.weights         = np.ones(self.n_assets, dtype=np.float32) / self.n_assets
        self.portfolio_value = 1.0
        self.value_history   = [1.0]
        self.turnover_history = []
        return self._get_obs(), {}

    def step(self, action):
        action  = np.exp(action - action.max())
        action  = action / action.sum()

        turnover = float(np.abs(action - self.weights).sum())
        tc_cost  = self.tc * turnover

        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((action * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = action.copy()
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret, 'turnover': turnover, 'tc_cost': tc_cost}
        return self._get_obs(), reward, done, False, info


## 4. 버킷 분할 — 엄격 분할(진짜 미노출 valid) vs 기존(노출) 분할

In [6]:
def bucket_to_ts(b):
    return pd.Timestamp(b * BUCKET_SEC, unit='s', tz='UTC')

def bucket_to_date(b):
    return bucket_to_ts(b).strftime('%Y-%m-%d')

full_train_buckets = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= RL_TRAIN_END]
true_train_buckets = [b for b in full_train_buckets if bucket_to_date(b) <= TRUE_TRAIN_END]
true_valid_buckets = [b for b in full_train_buckets if TRUE_VALID_START <= bucket_to_date(b) <= TRUE_VALID_END]
test_buckets        = [b for b in all_buckets if RL_TEST_START <= bucket_to_date(b) <= RL_TEST_END]

print(f'엄격 train(1~9월): {bucket_to_ts(true_train_buckets[0])} ~ {bucket_to_ts(true_train_buckets[-1])} ({len(true_train_buckets):,}스텝)')
print(f'진짜 미노출 valid(10~12월): {bucket_to_ts(true_valid_buckets[0])} ~ {bucket_to_ts(true_valid_buckets[-1])} ({len(true_valid_buckets):,}스텝)')
print(f'test(2026~): {bucket_to_ts(test_buckets[0])} ~ {bucket_to_ts(test_buckets[-1])} ({len(test_buckets):,}스텝)')
print('\n(참고: 기존 배포 모델은 full_train_buckets 전체로 학습됐고, 여기 true_valid_buckets도 그때는 학습에 포함돼 있었음)')

env_kwargs = dict(
    symbols=symbols, emb_index=emb_index, embs_all=embs_all,
    ret_index=ret_index, emb_dim=EMB_DIM, tc=TC,
)

print('\n환경 유효성 검사...')
check_env(CryptoPortfolioEnv(true_train_buckets[:100], **env_kwargs), warn=True)
print('OK')


엄격 train(1~9월): 2025-01-01 00:00:00+00:00 ~ 2025-09-30 23:30:00+00:00 (13,104스텝)
진짜 미노출 valid(10~12월): 2025-10-01 00:00:00+00:00 ~ 2025-12-31 23:30:00+00:00 (4,416스텝)
test(2026~): 2026-01-01 00:00:00+00:00 ~ 2026-07-14 03:30:00+00:00 (9,320스텝)

(참고: 기존 배포 모델은 full_train_buckets 전체로 학습됐고, 여기 true_valid_buckets도 그때는 학습에 포함돼 있었음)

환경 유효성 검사...
OK


## 5. Stage A — 시장 레짐 분할 (엄격 train 구간에서만 계산)

In [7]:
REGIME_NAMES = {1: 'bear', 2: 'pullback', 3: 'sideways', 4: 'rally', 5: 'bull'}
_DTW_MAX_LEN = 200


@dataclass
class MarketSegments:
    labels: pd.Series
    chunks: pd.DataFrame


def _zscore(x):
    std = x.std()
    return (x - x.mean()) / std if std > 1e-12 else np.zeros_like(x)


def _downsample(x, max_len=_DTW_MAX_LEN):
    if len(x) <= max_len:
        return x
    idx = np.linspace(0, len(x) - 1, max_len).round().astype(int)
    return x[idx]


def _dtw_distance(a, b):
    a, b = _downsample(a), _downsample(b)
    n, m = len(a), len(b)
    cost = np.full((n + 1, m + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, n + 1):
        ai = a[i - 1]
        row, prev = cost[i], cost[i - 1]
        for j in range(1, m + 1):
            row[j] = abs(ai - b[j - 1]) + min(prev[j], row[j - 1], prev[j - 1])
    return float(cost[n, m])


def _slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x), dtype=np.float64)
    a, _ = np.polyfit(t, x, 1)
    return float(a)


def _find_extrema_indices(x):
    d = np.diff(x)
    sign = np.sign(d)
    sign[sign == 0] = 1
    change = np.where(np.diff(sign) != 0)[0] + 1
    idx = np.concatenate(([0], change, [len(x) - 1]))
    return np.unique(idx)


def _merge_pass(x, bounds, merge_quantile):
    if len(bounds) <= 2:
        return bounds, False
    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
    if len(seg_ranges) < 2:
        return bounds, False
    slope_diffs = [abs(slopes[i + 1] - slopes[i]) for i in range(len(slopes) - 1)]
    dtw_dists = [
        _dtw_distance(
            _zscore(x[seg_ranges[i][0]:seg_ranges[i][1] + 1]),
            _zscore(x[seg_ranges[i + 1][0]:seg_ranges[i + 1][1] + 1]),
        )
        for i in range(len(seg_ranges) - 1)
    ]
    slope_thresh = np.quantile(slope_diffs, merge_quantile)
    dtw_thresh = np.quantile(dtw_dists, merge_quantile)

    new_bounds = [bounds[0]]
    merged_any = False
    i = 0
    while i < len(seg_ranges):
        can_merge = (
            i < len(seg_ranges) - 1
            and slope_diffs[i] <= slope_thresh
            and dtw_dists[i] <= dtw_thresh
        )
        if can_merge:
            new_bounds.append(seg_ranges[i + 1][1])
            merged_any = True
            i += 2
        else:
            new_bounds.append(seg_ranges[i][1])
            i += 1
    return new_bounds, merged_any


def _absorb_short_segments(x, bounds, min_length):
    while len(bounds) > 2:
        seg_ranges = list(zip(bounds[:-1], bounds[1:]))
        lengths = [b - a + 1 for a, b in seg_ranges]
        shortest = int(np.argmin(lengths))
        if lengths[shortest] >= min_length:
            break
        slopes = [_slope(x[a:b + 1]) for a, b in seg_ranges]
        left_diff = abs(slopes[shortest] - slopes[shortest - 1]) if shortest > 0 else np.inf
        right_diff = (
            abs(slopes[shortest] - slopes[shortest + 1]) if shortest < len(seg_ranges) - 1 else np.inf
        )
        drop_idx = shortest if left_diff <= right_diff else shortest + 1
        bounds = bounds[:drop_idx] + bounds[drop_idx + 1:]
    return bounds


def segment_and_label(series, n_labels=5, risk_threshold=0.2, lowpass_window=48,
                       merge_quantile=0.35, max_merge_iters=50, min_segment_length=None):
    x_raw = series.to_numpy(dtype=np.float64)
    if len(x_raw) < 3:
        raise ValueError('series가 너무 짧습니다 (최소 3개 포인트 필요)')
    if np.any(x_raw <= 0):
        raise ValueError('series는 로그 변환을 위해 모두 양수여야 합니다')

    x_log = np.log(x_raw)
    x_smooth = pd.Series(x_log).rolling(lowpass_window, min_periods=1, center=True).mean().to_numpy()

    bounds = _find_extrema_indices(x_smooth).tolist()
    for _ in range(max_merge_iters):
        bounds, merged = _merge_pass(x_smooth, bounds, merge_quantile)
        if not merged or len(bounds) <= 2:
            break
    bounds = _absorb_short_segments(x_log, bounds, min_segment_length or lowpass_window)

    seg_ranges = list(zip(bounds[:-1], bounds[1:]))
    slopes = np.array([_slope(x_log[a:b + 1]) for a, b in seg_ranges])

    theta = risk_threshold
    H = np.quantile(slopes, 1 - theta / 2)
    L = np.quantile(slopes, theta / 2)

    labels = np.empty(len(slopes), dtype=np.int64)
    for i, s in enumerate(slopes):
        if s > H:
            labels[i] = n_labels
        elif s < L:
            labels[i] = 1
        elif H > L:
            frac = (s - L) / (H - L)
            labels[i] = int(np.clip(2 + frac * (n_labels - 2), 2, n_labels - 1))
        else:
            labels[i] = (n_labels + 1) // 2

    per_point_labels = np.empty(len(x_raw), dtype=np.int64)
    for (a, b), lbl in zip(seg_ranges, labels):
        per_point_labels[a:b + 1] = lbl

    labels_series = pd.Series(per_point_labels, index=series.index, name='regime_label')

    chunks = pd.DataFrame({
        'start_idx': [a for a, _ in seg_ranges],
        'end_idx': [b for _, b in seg_ranges],
        'length': [b - a + 1 for a, b in seg_ranges],
        'slope': slopes,
        'label': labels,
    })
    chunks['regime'] = chunks['label'].map(REGIME_NAMES) if n_labels == 5 else chunks['label'].astype(str)
    return MarketSegments(labels=labels_series, chunks=chunks)


def chunk_sampling_priority(chunks, beta, risk_threshold=0.2):
    r = chunks['slope'].to_numpy(dtype=np.float64)
    theta = risk_threshold
    Ht = np.quantile(r, 1 - theta / 2)
    Lt = np.quantile(r, theta / 2)

    std = r.std() if r.std() > 1e-12 else 1.0
    h = max(0.9 * std * len(r) ** (-1 / 5), 1e-8)
    diffs = (r[:, None] - r[None, :]) / h
    pdf = np.exp(-0.5 * diffs ** 2).sum(axis=1) / (len(r) * h * np.sqrt(2 * np.pi))
    pdf = np.maximum(pdf, 1e-12)

    tail_mask = (r >= Ht) | (r <= Lt)
    log_w = beta * r - np.where(tail_mask, 0.0, np.log(pdf))
    log_w -= log_w.max()
    weights = np.exp(log_w)
    return weights / weights.sum()


print('market_segmentation 함수 정의 완료')


market_segmentation 함수 정의 완료


In [13]:
# Stage A는 엄격 train 구간(1~9월)에서만 계산 — valid(10~12월)를 조금이라도 참조하지 않도록
eq_ret_per_bucket = np.array([
    np.mean([ret_index.get((sym, b), 0.0) for sym in symbols])
    for b in true_train_buckets
])
eq_nav = pd.Series(np.cumprod(1 + eq_ret_per_bucket), index=range(len(true_train_buckets)))

seg = segment_and_label(
    eq_nav, n_labels=N_LABELS, risk_threshold=RISK_THRESHOLD,
    lowpass_window=LOWPASS_WINDOW,
)
print(f"구간 수: {len(seg.chunks)} (평균 길이 {seg.chunks['length'].mean():.0f}버킷 ≈ {seg.chunks['length'].mean()/48:.1f}일)")
print(seg.chunks['regime'].value_counts())


구간 수: 84 (평균 길이 157버킷 ≈ 3.3일)
regime
rally       32
sideways    18
pullback    16
bear         9
bull         9
Name: count, dtype: int64


## 6. 저수준 환경 — SparseGatedCryptoPortfolioEnv + ChunkedSparseGatedEnv

In [14]:
class SparseGatedCryptoPortfolioEnv(CryptoPortfolioEnv):
    def __init__(self, *args, top_k=TOP_K, **kwargs):
        self.top_k = top_k
        super().__init__(*args, **kwargs)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(self.n_assets + 1,), dtype=np.float32)

    def action_to_weights(self, action):
        action = np.asarray(action, dtype=np.float64)
        asset_logits, gate_logit = action[:-1], action[-1]
        k = min(self.top_k, self.n_assets)
        top_idx = np.argpartition(asset_logits, -k)[-k:]
        sel = asset_logits[top_idx]
        sel = np.exp(sel - sel.max())
        sel = sel / sel.sum()
        risky_w = np.zeros(self.n_assets, dtype=np.float32)
        risky_w[top_idx] = sel
        rho = float(1.0 / (1.0 + np.exp(-gate_logit)))
        return (rho * risky_w).astype(np.float32), rho, top_idx

    def reset(self, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.rho_history = []
        self.selection_history = []
        return obs, info

    def step_with_weights(self, target_w, extra_info=None):
        turnover = float(np.abs(target_w - self.weights).sum())
        tc_cost  = self.tc * turnover
        b          = self.buckets[self.t]
        asset_rets = self._asset_ret(b)
        port_ret   = float((target_w * asset_rets).sum())

        self.portfolio_value *= (1 + port_ret - tc_cost)
        self.weights           = target_w.copy()
        self.value_history.append(self.portfolio_value)
        self.turnover_history.append(turnover)
        self.t                += 1

        done = (self.t >= len(self.buckets) - 1)
        reward = float(port_ret - tc_cost)
        info = {'portfolio_value': self.portfolio_value, 'port_ret': port_ret,
                'turnover': turnover, 'tc_cost': tc_cost}
        if extra_info:
            info.update(extra_info)
            if 'rho' in extra_info:
                self.rho_history.append(extra_info['rho'])
            if 'selected' in extra_info:
                self.selection_history.append(extra_info['selected'])
        return self._get_obs(), reward, done, False, info

    def step(self, action):
        target_w, rho, top_idx = self.action_to_weights(action)
        return self.step_with_weights(target_w, extra_info={'rho': rho, 'selected': top_idx.tolist()})


class ChunkedSparseGatedEnv(SparseGatedCryptoPortfolioEnv):
    def __init__(self, all_buckets, chunks, beta, risk_threshold=0.2, rng=None, top_k=TOP_K, **env_kwargs):
        self.all_buckets = all_buckets
        self.chunks      = chunks.reset_index(drop=True)
        self.beta        = beta
        self.priority    = chunk_sampling_priority(self.chunks, beta, risk_threshold)
        self.rng         = rng or np.random.default_rng()
        super().__init__(buckets=all_buckets[:2], top_k=top_k, **env_kwargs)

    def reset(self, seed=None, options=None):
        idx = self.rng.choice(len(self.chunks), p=self.priority)
        row = self.chunks.iloc[idx]
        self.buckets = self.all_buckets[int(row['start_idx']):int(row['end_idx']) + 1]
        return super().reset(seed=seed, options=options)


## 7. 성과 계산 함수

In [15]:
def compute_perf(values, steps_per_year=48 * 365):
    rets       = np.diff(values) / values[:-1]
    total_ret  = values[-1] / values[0] - 1
    annual_ret = (1 + total_ret) ** (steps_per_year / len(rets)) - 1
    sharpe     = rets.mean() / (rets.std() + 1e-8) * np.sqrt(steps_per_year)
    max_dd     = ((values / np.maximum.accumulate(values)) - 1).min()
    return {'총 수익률': total_ret, '연환산 수익률': annual_ret, 'Sharpe': sharpe, 'MDD': max_dd}


def perf_row(values, turnovers, tc=TC):
    perf = compute_perf(values)
    t = np.asarray(turnovers, dtype=np.float64)
    perf['평균 턴오버/스텝'] = t.mean()
    perf['누적 비용 드래그'] = 1 - np.prod(1 - tc * t)
    return perf


def equal_weight_backtest(env):
    obs, _ = env.reset()
    done, values = False, [1.0]
    action = np.ones(env.n_assets, dtype=np.float32) / env.n_assets
    while not done:
        obs, _, done, _, info = env.step(action)
        values.append(info['portfolio_value'])
    return np.array(values), list(env.turnover_history)


def run_beta_smoothed(model, alpha, buckets, top_k=TOP_K):
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=top_k, **env_kwargs)
    obs, _ = env.reset()
    done = False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        target_w, rho, top_idx = env.action_to_weights(a)
        w = (1 - alpha) * env.weights + alpha * target_w
        obs, _, done, _, info = env.step_with_weights(w, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return np.array(env.value_history), list(env.turnover_history)


## 8. β=-30 재학습 — 2025년 1~9월만 (seed=42, §13-C와 동일 예산·하이퍼파라미터)

In [16]:
%%time
CLEAN_MODEL_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
chunk_env = Monitor(ChunkedSparseGatedEnv(
    all_buckets=true_train_buckets, chunks=seg.chunks, beta=BETA,
    risk_threshold=RISK_THRESHOLD, rng=rng, top_k=TOP_K, **env_kwargs,
))
clean_model = PPO(
    'MlpPolicy', chunk_env,
    learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10,
    gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5,
    max_grad_norm=0.5, policy_kwargs=dict(net_arch=[512, 256]),
    verbose=0, device='cpu', seed=SEED,
)
clean_model.learn(total_timesteps=POOL_TIMESTEPS_EACH, progress_bar=True)

clean_model_path = CLEAN_MODEL_DIR / f'agent_beta{BETA}_seed{SEED}_train0109.zip'
clean_model.save(str(clean_model_path))
print(f'저장: {clean_model_path.name}')


Output()

저장: agent_beta-30_seed42_train0109.zip
CPU times: user 16min 27s, sys: 5.94 s, total: 16min 33s
Wall time: 16min 57s


In [17]:
# 재실행용 로더 (idempotent)
if 'clean_model' not in globals():
    clean_model_path = CLEAN_MODEL_DIR / f'agent_beta{BETA}_seed{SEED}_train0109.zip'
    clean_model = PPO.load(str(clean_model_path))
    print('클린 모델 로드 완료')


## 9. α 스윕 — 진짜 미노출 valid(10~12월)에서

In [18]:
%%time
print('=== 클린 모델(1~9월 학습) — 진짜 미노출 valid(10~12월)에서 α 스윕 ===\n')

clean_valid_net_by_alpha = {}
for alpha in SMOOTH_ALPHAS:
    net, turns = run_beta_smoothed(clean_model, alpha, true_valid_buckets)
    perf = perf_row(net, turns)
    label = 'α=1.0(스무딩 없음)' if alpha == 1.0 else f'α={alpha}'
    print(f'{label}: 총수익 {perf["총 수익률"]*100:.2f}% | Sharpe {perf["Sharpe"]:.3f}')
    clean_valid_net_by_alpha[alpha] = net[-1]

CLEAN_BEST_ALPHA = max((a for a in clean_valid_net_by_alpha if a < 1.0), key=lambda a: clean_valid_net_by_alpha[a])
print(f'\n[진짜 미노출 valid에서 선택] α = {CLEAN_BEST_ALPHA} (valid 총수익률 {(clean_valid_net_by_alpha[CLEAN_BEST_ALPHA]-1)*100:.2f}%)')


=== 클린 모델(1~9월 학습) — 진짜 미노출 valid(10~12월)에서 α 스윕 ===

α=1.0(스무딩 없음): 총수익 -15.32% | Sharpe -2.148
α=0.3: 총수익 -8.60% | Sharpe -1.107
α=0.15: 총수익 -5.30% | Sharpe -0.597
α=0.05: 총수익 -1.05% | Sharpe -0.003
α=0.03: 총수익 0.57% | Sharpe 0.219
α=0.02: 총수익 1.70% | Sharpe 0.373
α=0.01: 총수익 3.10% | Sharpe 0.560

[진짜 미노출 valid에서 선택] α = 0.01 (valid 총수익률 3.10%)
CPU times: user 39.1 s, sys: 89.9 ms, total: 39.2 s
Wall time: 39.7 s


## 10. test는 1회만 — 기존 배포 모델과 나란히 비교

In [19]:
print(f'[클린 모델] 선택된 α={CLEAN_BEST_ALPHA}로 test 1회 실행...')
clean_test_net, clean_test_turns = run_beta_smoothed(clean_model, CLEAN_BEST_ALPHA, test_buckets)
clean_test_perf = perf_row(clean_test_net, clean_test_turns)

print('[기존 배포 모델] 참고용 재실행 (2025 전체 학습, 노출된 valid에서 α=0.01 선택)...')
deployed_model = PPO.load(str(DEPLOYED_AGENT_PATH))
deployed_test_net, deployed_test_turns = run_beta_smoothed(deployed_model, 0.01, test_buckets)
deployed_test_perf = perf_row(deployed_test_net, deployed_test_turns)

ew_test_values, ew_test_turnovers = equal_weight_backtest(CryptoPortfolioEnv(test_buckets, **env_kwargs))
ew_test_perf = perf_row(ew_test_values, ew_test_turnovers)

comparison = pd.DataFrame({
    '등가중 벤치마크': ew_test_perf,
    f'기존 배포(2025 전체 학습, α=0.01)': deployed_test_perf,
    f'클린(1~9월 학습, valid에서 α={CLEAN_BEST_ALPHA} 선택)': clean_test_perf,
}).T

print()
print(comparison.to_string(float_format=lambda x: f'{x:.4f}'))
comparison.to_csv(CRYPTO_DIR / 'crypto_clean_valid_comparison.csv', encoding='utf-8-sig')
print('\n저장 완료: crypto_clean_valid_comparison.csv')


[클린 모델] 선택된 α=0.01로 test 1회 실행...
[기존 배포 모델] 참고용 재실행 (2025 전체 학습, 노출된 valid에서 α=0.01 선택)...

                                총 수익률  연환산 수익률  Sharpe     MDD  평균 턴오버/스텝  누적 비용 드래그
등가중 벤치마크                       0.0266   0.0505  0.3672 -0.4009     0.0000     0.0000
기존 배포(2025 전체 학습, α=0.01)      0.0566   0.1091  1.1556 -0.0743     0.0011     0.0099
클린(1~9월 학습, valid에서 α=0.01 선택) 0.0783   0.1522  1.3931 -0.0873     0.0008     0.0072

저장 완료: crypto_clean_valid_comparison.csv


## 11. 판정 기준

- **클린 모델의 test 성적이 기존 배포 모델과 비슷하거나 크게 안 떨어진다** → 지금까지 보고한
  성능이 "valid가 사실 노출돼 있었다"는 문제 때문에 부풀려진 게 아니라는 강한 근거 확보 —
  `paper/GAP_ANALYSIS.md` §1.3을 완료 처리하고, 이 결과를 논문의 핵심 강건성 증거로 사용.
- **클린 모델이 크게 못하다(예: 등가중 근처거나 그 이하)** → 지금까지 보고한 성능(예: 배포 모델
  test +5.66%)의 상당 부분이 valid 노출에서 온 과최적화였다는 뜻 — 정직하게 그렇게 보고해야 하고,
  논문의 "진짜 알파" 주장 전체를 재검토해야 함(9개월치 데이터만으로 학습한 게 부족했을 가능성도
  별도로 고려 — 학습량 자체의 영향과 valid 노출의 영향을 이 실험 하나로는 완전히 분리 못 함을
  한계로 명시할 것).
- 어느 쪽이든, 이게 이 프로젝트에서 나온 **가장 깨끗한(선택 편향 없는) 단일 test 관측치**이므로
  논문에 반드시 포함할 것.

결과를 `paper/FINDINGS_LOG.md`에 추가하고 `paper/GAP_ANALYSIS.md` §1.3을 갱신할 것.
